## Define a Ladder Operator

#### Import

In [1]:
import src.bosehubbard.bosehubbard as bh
import numpy as np
import scipy
import scipy.sparse as sparse
from pyexpokit import expmv
from tqdm import tqdm
import matplotlib.pyplot as plt
from src.bosehubbard.utils.lieb import lieb_lattice,occupation_number
from typing import List,Callable

#### Data

In [2]:
linear_dimension:int=9
hopping_energy:float=1.
interaction_coupling:float=10.

links:List=[[i, (i+1) % linear_dimension,1] for i in range(linear_dimension)]
u=0.
omegas=[0.]*linear_dimension

model = bh.Model(omegas, links, u)

model_two_particle=model.numbersector(2)
hamiltonian=model_two_particle.hamiltonian

(45, 45)


#### $a^{\dagger}_i a_j$ operator

In [3]:
model_two_particle.basis.vs[0, :]
print(model_two_particle.basis.vs.shape)

def adag_a(i:int,j:int,psi:np.ndarray,basis:Callable)->np.ndarray:
    idx=np.nonzero(psi)
    new_psi=np.zeros_like(psi)
    for k in idx:
        k=k[0]
        print(k)
        print(basis.vs[k])
        if basis.vs[k,j]!=0 and basis.vs[k,i]!=np.sum(basis.vs[k]):
            psi_value=psi[k]*np.sqrt(basis.vs[k,j]*(basis.vs[k,i]+1))
            new_basis=basis.vs[k].copy()
            new_basis[i]=basis.vs[k,i]+1
            new_basis[j]=basis.vs[k,j]-1
            print(new_basis)
            new_index=basis.index(new_basis)
            print(new_index)
            new_psi[new_index]=psi_value
    return new_psi

(45, 9)


#### Let's try it

Initialize the state

In [4]:
psi0=np.zeros(hamiltonian.shape[0])

# we need to find the indices related to the 2 particles in the site 24-th
state:np.ndarray=np.zeros(linear_dimension)
state[5]=2

print(state)
index=model_two_particle.basis.index(state) # random initial site
print(index)
psi0[index]=1.
psi0=psi0/np.linalg.norm(psi0)

[0. 0. 0. 0. 0. 2. 0. 0. 0.]
35


In [6]:
adag_a_psi0=adag_a(5,4,psi0,model_two_particle.basis)
print(psi0)
print(adag_a_psi0)
idx=np.nonzero(adag_a_psi0)
print(idx)

35
[0 0 0 0 0 2 0 0 0]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
(array([], dtype=int64),)
